In [5]:
import pandas as pd
import time
import requests
import zipfile
import os
import json
import glob
import csv

In [6]:

base_url = "https://opentender.net/api/tender/export-ocds-batch/"
years = list(range(2015, 2024))

all_data = []
years

[2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]

In [7]:
extract_folder = "../data/json"
os.makedirs(extract_folder, exist_ok=True)

for year in years:
    target_url = f"{base_url}?year={year}&lpse=127"
    print(f"Downloading: Year {year}...")
    
    try:
        res = requests.get(target_url, stream=True)
        
        zip_filename = f"ocds_{year}.zip"
        
        # save zip
        with open(zip_filename, "wb") as f:
            for chunk in res.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
        
        print(f"Saved: {zip_filename}")
        
        # unzip + rename langsung
        with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
            for member in zip_ref.namelist():
                if member.endswith(".json"):
                    
                    # extract file
                    zip_ref.extract(member, extract_folder)
                    
                    filename = os.path.basename(member)
                    old_path = os.path.join(extract_folder, filename)
                    
                    # nama baru per tahun
                    new_name = f"ocds-data-tender-{year}.json"
                    new_path = os.path.join(extract_folder, new_name)
                    
                    # overwrite aman (1 file per year)
                    if os.path.exists(new_path):
                        os.remove(new_path)
                    
                    os.rename(old_path, new_path)
                    
                    print(f"Renamed → {new_name}")
        
        print(f"Done Year {year}")
        
        os.remove(zip_filename)
        time.sleep(3)
        
    except Exception as e:
        print(f"Error for {year}: {e}")

Downloading: Year 2015...
Saved: ocds_2015.zip
Renamed → ocds-data-tender-2015.json
Done Year 2015
Downloading: Year 2016...
Saved: ocds_2016.zip
Renamed → ocds-data-tender-2016.json
Done Year 2016
Downloading: Year 2017...
Saved: ocds_2017.zip
Renamed → ocds-data-tender-2017.json
Done Year 2017
Downloading: Year 2018...
Saved: ocds_2018.zip
Renamed → ocds-data-tender-2018.json
Done Year 2018
Downloading: Year 2019...
Saved: ocds_2019.zip
Renamed → ocds-data-tender-2019.json
Done Year 2019
Downloading: Year 2020...
Saved: ocds_2020.zip
Renamed → ocds-data-tender-2020.json
Done Year 2020
Downloading: Year 2021...
Saved: ocds_2021.zip
Renamed → ocds-data-tender-2021.json
Done Year 2021
Downloading: Year 2022...
Saved: ocds_2022.zip
Renamed → ocds-data-tender-2022.json
Done Year 2022
Downloading: Year 2023...
Saved: ocds_2023.zip
Renamed → ocds-data-tender-2023.json
Done Year 2023


Convert json to CSV file

In [8]:
path_folder = '../data/json'
output_file = '../data/csv/all_data_ocds.csv'

os.makedirs('../data/csv', exist_ok=True)

files = glob.glob(os.path.join(path_folder, "ocds-data-tender-*.json"))

fieldnames = [
    'ocid', 'release_id', 'date', 'buyer_name', 'buyer_id',
    'tender_id', 'tender_title', 'mainProcurementCategory',
    'tender_minValue', 'tender_datePublished', 'tender_status',
    'award_id', 'award_title', 'award_date', 'award_value', 'award_supplier'
]

with open(output_file, 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()

    if not files:
        print("Folder kosong atau file tidak ditemukan!")
    else:
        total_rows = 0

        for file_path in files:
            print(f"Processing: {file_path}")

            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)

                for release in data.get('releases', []):
                    tender = release.get('tender', {})
                    buyer = release.get('buyer', {})
                    awards = release.get('awards', [])

                    for award in awards:
                        suppliers = award.get('suppliers', [])
                        supplier_names = ', '.join([s.get('name', '') for s in suppliers]) or None

                        writer.writerow({
                            'ocid': release.get('ocid'),
                            'release_id': release.get('id'),
                            'date': release.get('date'),
                            'buyer_name': buyer.get('name'),
                            'buyer_id': buyer.get('id'),
                            'tender_id': tender.get('id'),
                            'tender_title': tender.get('title'),
                            'mainProcurementCategory': tender.get('mainProcurementCategory'),
                            'tender_minValue': tender.get('minValue', {}).get('amount'),
                            'tender_datePublished': tender.get('datePublished'),
                            'tender_status': tender.get('status'),
                            'award_id': award.get('id'),
                            'award_title': award.get('title'),
                            'award_date': award.get('date'),
                            'award_value': award.get('value', {}).get('amount'),
                            'award_supplier': supplier_names
                        })

                        total_rows += 1

            except Exception as e:
                print(f"Gagal membaca {file_path}: {e}")

        print(f"Selesai! Total data: {total_rows} baris")

Processing: ../data/json\ocds-data-tender-2015.json
Processing: ../data/json\ocds-data-tender-2016.json
Processing: ../data/json\ocds-data-tender-2017.json
Processing: ../data/json\ocds-data-tender-2018.json
Processing: ../data/json\ocds-data-tender-2019.json
Processing: ../data/json\ocds-data-tender-2020.json
Processing: ../data/json\ocds-data-tender-2021.json
Processing: ../data/json\ocds-data-tender-2022.json
Processing: ../data/json\ocds-data-tender-2023.json
Selesai! Total data: 8678 baris
